# 실기 대비
# 실전 문제풀이
# set 3

## 1) 데이터 및 시나리오

### 시중 스마트폰 상세 정보

> 갓 입사한 신입사원 김모씨는 기획 부서에 배치되었다. 첫 프로젝트로 신규 스마트폰 스펙 기획 업무를 보조하게 되었다. 장고 끝에 김씨는 온라인 디지털 마켓 사이트 `디판다요`에서 판매중인 스마트폰 데이터를 수집하여 이를 분석하고 향후 프로젝트의 빅데이터로 활용하기로 결심하였다.

### 데이터 개요

| 파일명 | 행 | 열 | 인코딩 |
|---|---:|---:|---|
| `mobiles.csv` | 430 | 11 | UTF-8 |

## 1) 데이터 및 시나리오

### 변수 상세

| 변수명 | 유형 | 설명 |
|---|---|---|
| `screen_size` | string | 화면 크기 |
| `ROM` | int | 저장 공간 용량 |
| `RAM` | int | RAM 용량 |
| `num_rear_camera` | int | 후면 카메라 개수 |
| `num_front_camera` | int | 전면 카메라 개수 |
| `battery_capacity` | int | 배터리 용량 |
| `ratings` | float | 평가 점수 평균 |
| `num_of_ratings` | int | 평가 개수 |
| `sales_price` | int | 판매가격 |
| `discount_percent` | float | 할인율 |
| `sales` | float | 판매 지수 |

## 2) 문제

### 필요 라이브러리 함수 및 클래스 목록

| 목록 |
|---|
| `from sklearn.preprocessing import MinMaxScaler` |
| `from sklearn.model_selection import train_test_split` |
| `from sklearn.neighbors import KNeighborsRegressor` |
| `from sklearn.metrics import mean_squared_error` |

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

In [4]:
df = pd.read_csv('../../dataset/mobiles.csv')
#SCDI
display(df.shape)
display(df.columns)
display(df.dtypes)
display(df.isna().sum())

(430, 11)

Index(['screen_size', 'ROM', 'RAM', 'num_rear_camera', 'num_front_camera',
       'battery_capacity', 'ratings', 'num_of_ratings', 'sales_price',
       'discount_percent', 'sales'],
      dtype='object')

screen_size          object
ROM                   int64
RAM                   int64
num_rear_camera       int64
num_front_camera      int64
battery_capacity      int64
ratings             float64
num_of_ratings        int64
sales_price           int64
discount_percent    float64
sales               float64
dtype: object

screen_size         0
ROM                 0
RAM                 0
num_rear_camera     0
num_front_camera    0
battery_capacity    0
ratings             0
num_of_ratings      0
sales_price         0
discount_percent    0
sales               0
dtype: int64

### Q01.

스마트폰의 경우 많은 제품이 출시되지만 정작 주목받는 제품은 극히 적다고 한다.  
판매지수(`sales`)를 기준으로 이상치라고 판단되는 제품을 주목받는 제품이라고 판단하고 해당 제품들의 성능지표를 산출하시오.  
산출된 성능지표의 평균은 얼마인가?

#### 성능지표 계산식

$$
\text{성능지표}
=
\frac{ROM}{32}
+
\frac{RAM}{2}
+
\text{카메라 개수}
+
\frac{\text{배터리 용량}(battery\_capacity)}{1000}
$$

※ 카메라 개수: `num_rear_camera + num_front_camera`  
※ 이상치는 평균으로부터 2 표준편차보다 큰 값으로 정의한다.  
※ 결과는 반올림하여 소수점 둘째 자리까지 계산하시오. `(정답 예시: 0.12)`

In [36]:
df_q1 = df.copy()
sales_mean = df_q1['sales'].mean()
sales_std = df_q1['sales'].std()
display(sales_mean, sales_std)

cond_q1 = (df_q1['sales'] - sales_mean) > 2*sales_std #이전문제풀이 성공했는데 절대값으로 해야하는 거 아닌가? 아님 
display(cond_q1.value_counts())

df_q1['r-r-c-b'] = df_q1['ROM']/32 + df_q1['RAM']/2 + (df_q1['num_rear_camera'] + df_q1['num_front_camera']) + df_q1['battery_capacity']/1000 
df_q1_1 = df_q1.loc[cond_q1, :].copy()
display(df_q1_1.shape)

round(df_q1_1['r-r-c-b'].mean(), 2)

29.75232558139536

58.39958785566842

False    414
True      16
Name: sales, dtype: int64

(16, 12)

11.01

### Q02.

판매 지수(`sales`)와 가장 상관관계가 높은 변수를 찾고자 한다.  
배터리 용량(`battery_capacity`), 평가 점수 평균(`ratings`), 평가 개수(`num_of_ratings`), 판매 가격(`sales_price`), 할인율(`discount_percent`) 변수와 판매 지수(`sales`)를 피어슨 상관분석을 실시하였을 때 상관계수의 절대값이 가장 큰 변수의 상관계수는 얼마인가?

※ 후면 카메라가 1개인 제품은 제외하시오.  
※ 통계적 유의성은 고려하지 않음.  
※ 결과는 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [19]:
df_q2 = df.copy()
display(df_q2.shape)
cols_q2 = ['battery_capacity', 'ratings', 'num_of_ratings', 'sales_price', 'discount_percent', 'sales']
df_q2 = df_q2.loc[~(df_q2['num_rear_camera'] == 1), :].copy()
display(df_q2.shape)
df_q2_1 = df_q2[cols_q2]
display(df_q2_1)

ser_q2 = df_q2_1.corr(method='pearson')['sales'].drop(index='sales').abs()
display(ser_q2)

display(round(ser_q2.max(),2), ser_q2.idxmax())


(430, 11)

(390, 11)

,battery_capacity,ratings,num_of_ratings,sales_price,discount_percent,sales
1,2815,4.5,244,57149,0.04,1.39
4,2815,4.6,745,69149,0.02,5.15
5,2815,4.6,745,64149,0.02,4.78
6,2815,4.6,745,69149,0.02,5.15
7,2815,4.6,745,64149,0.02,4.78
...,...,...,...,...,...,...
425,4000,4.3,1870,7999,0.30,1.50
426,4000,4.3,1783,9699,0.28,1.73
427,4250,4.2,1554,21999,0.12,3.42
428,5000,4.2,8161,8299,0.07,6.77


battery_capacity    0.025680
ratings             0.226075
num_of_ratings      0.949114
sales_price         0.247760
discount_percent    0.223471
Name: sales, dtype: float64

0.95

'num_of_ratings'

### Q03.

판매 지수(`sales`)를 예측하기 위해 머신러닝 모델을 활용하고자 한다.  
k-NN 알고리즘을 사용하고 연산에 사용하는 이웃의 개수를 변화하면서 가장 성능이 좋은 모델을 확보하려 한다.  
RMSE(Root Mean Squared Error)를 기준으로 가장 성능이 좋은 모델을 확인하고 해당 모델의 k(이웃 개수)를 구하라.

#### 독립변수 / 종속변수

| 구분 | 변수 |
|---|---|
| 독립변수 | 판매 지수를 제외한 모든 변수 |
| 종속변수 | 판매 지수 |

※ 명목형 독립변수의 경우 One Hot Encoding을 실시한 결과를 모델에 사용하시오.  
※ 학습에 사용하는 독립변수의 개수는 총 14개이다.  
※ 학습 및 평가 데이터 세트 분할비는 8:2로 하시오.  
※ 정규화는 Min-Max 정규화를 실시하며 평가 데이터 세트는 학습 데이터 세트 기반으로 정규화 하시오.  
※ seed는 `123`으로 고정하시오.  
※ 최근접 이웃은 3, 5, 7, 9, 11개를 사용하시오.  
※ 정답은 자연수로 출력하시오. `(정답 예시: 7)`

In [35]:
df_q3 = df.copy()
display(df_q3.shape)

#D
X = df_q3.drop(columns=['sales'])
y = df_q3['sales'] #series, 독립변수는 일반적으로 ser 여도 무방
display(X.shape,y.shape)

display(X.dtypes)
X_ohe = pd.get_dummies(X, columns=['screen_size'])
display(X_ohe.shape, X_ohe.columns)

train_X, test_X, train_y, test_y = train_test_split(X_ohe, y, train_size=0.8, random_state = 123)
#N
scaler = MinMaxScaler()
train_X_n = scaler.fit_transform(train_X)
test_X_n = scaler.transform(test_X)

#M#E
K = [3,5,7,9,11]

dict_knn = {}
for k in K :
    model = KNeighborsRegressor(n_neighbors = k) #randoms state를 안쓰는 이유?
    model.fit(train_X_n,train_y)
    pred_y = model.predict(test_X_n)
    dict_knn[k] = mean_squared_error(test_y,pred_y)**0.5
display(dict_knn)

ser_q3 = pd.Series(dict_knn)
display(ser_q3)
display(ser_q3.idxmin())
    

(430, 11)

(430, 10)

(430,)

screen_size          object
ROM                   int64
RAM                   int64
num_rear_camera       int64
num_front_camera      int64
battery_capacity      int64
ratings             float64
num_of_ratings        int64
sales_price           int64
discount_percent    float64
dtype: object

(430, 14)

Index(['ROM', 'RAM', 'num_rear_camera', 'num_front_camera', 'battery_capacity',
       'ratings', 'num_of_ratings', 'sales_price', 'discount_percent',
       'screen_size_Large', 'screen_size_Medium', 'screen_size_Small',
       'screen_size_Very Large', 'screen_size_Very Small'],
      dtype='object')

{3: 40.440548901789604,
 5: 48.80082671049649,
 7: 53.18675529199676,
 9: 55.48438386829515,
 11: 56.16070310635331}

3     40.440549
5     48.800827
7     53.186755
9     55.484384
11    56.160703
dtype: float64

3